# UNSW-NB15 Intrusion Detection System (IDS): Model Training

## Overview
This notebook trains a supervised **AdaBoost (Adaptive Boosting)** classifier using the pre-split **UNSW-NB15** benchmark dataset from `datasets/UNSW-NB15_splited/`:
- **70% Training Set (`train.csv`)**: $N = 180,371$ rows used for model fitting.
- **Categorical Feature Encoding**: Encodes non-numeric columns (`proto`, `service`, `state`) using `OrdinalEncoder` fitted strictly on the training set.
- **Model Training**: Fits `AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=42)`.
- **Model Export**: Serializes the trained classifier, fitted encoder, and training metadata to `models/IDS/adaboost/{timestamp}/`.

In [1]:
import os
import json
import pickle
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier
from sklearn.preprocessing import OrdinalEncoder

# Visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment and libraries loaded successfully.")

Environment and libraries loaded successfully.


## 1. Load Pre-Split UNSW-NB15 Dataset

We load the pre-split dataset from `datasets/UNSW-NB15_splited/` generated by `notebooks/data_processing/unsw_nb15.ipynb`.

In [2]:
# Resolve path to UNSW-NB15_splited directory
base_dir = Path.cwd()
possible_dirs = [
    base_dir / '../../datasets/UNSW-NB15_splited',
    base_dir / 'datasets/UNSW-NB15_splited',
    Path(r'd:/Swinburne Courses/Project A/datasets/UNSW-NB15_splited')
]

splited_dir = next((d.resolve() for d in possible_dirs if d.resolve().exists()), None)
if splited_dir is None:
    raise FileNotFoundError("Could not locate datasets/UNSW-NB15_splited directory.")

print(f"Loading pre-split datasets from: {splited_dir}")
train_df = pd.read_csv(splited_dir / 'train.csv')
test_df = pd.read_csv(splited_dir / 'test.csv')
holdout_df = pd.read_csv(splited_dir / 'holdout.csv')

total_rows = len(train_df) + len(test_df) + len(holdout_df)

split_summary = pd.DataFrame({
    'Split': ['Training Set (70%)', 'Testing / Validation Set (15%)', 'Holdout Test Set (15%)', 'Total Dataset'],
    'File': ['train.csv', 'test.csv', 'holdout.csv', '-'],
    'Row Count': [len(train_df), len(test_df), len(holdout_df), total_rows],
    'Ratio (%)': [
        len(train_df) / total_rows * 100,
        len(test_df) / total_rows * 100,
        len(holdout_df) / total_rows * 100,
        100.0
    ]
})

print("\n=== Pre-Split Dataset Summary ===")
display(split_summary.style.format({'Row Count': '{:,}', 'Ratio (%)': '{:.2f}%'}))

Loading pre-split datasets from: D:\Swinburne Courses\Project A\datasets\UNSW-NB15_splited



=== Pre-Split Dataset Summary ===


,Split,File,Row Count,Ratio (%)
0,Training Set (70%),train.csv,"180,371",70.00%
1,Testing / Validation Set (15%),test.csv,"38,651",15.00%
2,Holdout Test Set (15%),holdout.csv,"38,651",15.00%
3,Total Dataset,-,"257,673",100.00%


## 2. Feature and Target Separation (X and y)

Separate features ($X$) from target labels:
- **Binary Target ($y_{bin}$)**: `label` (0 = Normal, 1 = Attack).
- **Multi-Class Target ($y_{multi}$)**: `attack_cat` (10 categories).

In [3]:
target_cols = ['label', 'attack_cat']
feature_cols = [c for c in train_df.columns if c not in target_cols]

print(f"Total Features: {len(feature_cols)}")

# Features
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
X_holdout = holdout_df[feature_cols]

# Binary targets
y_train_bin = train_df['label']
y_test_bin = test_df['label']
y_holdout_bin = holdout_df['label']

# Multi-class targets
y_train_multi = train_df['attack_cat']
y_test_multi = test_df['attack_cat']
y_holdout_multi = holdout_df['attack_cat']

print(f"X_train shape:   {X_train.shape}")
print(f"X_test shape:    {X_test.shape}")
print(f"X_holdout shape: {X_holdout.shape}")

Total Features: 52
X_train shape:   (180371, 52)
X_test shape:    (38651, 52)
X_holdout shape: (38651, 52)


## 3. Categorical Encoding & Feature Preprocessing

Encode non-numeric columns (`proto`, `service`, `state`) using `OrdinalEncoder` fitted strictly on `X_train` with `unknown_value=-1` to prevent data leakage.

In [4]:
cat_cols = ['proto', 'service', 'state']
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Categorical features to encode ({len(cat_cols)}): {cat_cols}")
print(f"Numerical features ({len(num_cols)}): {num_cols[:5]} ... (total {len(num_cols)})")

# Initialize and fit OrdinalEncoder strictly on training split
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train_enc = X_train.copy()
X_test_enc = X_test.copy()
X_holdout_enc = X_holdout.copy()

X_train_enc[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test_enc[cat_cols] = encoder.transform(X_test[cat_cols])
X_holdout_enc[cat_cols] = encoder.transform(X_holdout[cat_cols])

print(f"\nEncoded feature matrix shapes:")
print(f"X_train_enc:   {X_train_enc.shape}")
print(f"X_test_enc:    {X_test_enc.shape}")
print(f"X_holdout_enc: {X_holdout_enc.shape}")

Categorical features to encode (3): ['proto', 'service', 'state']
Numerical features (49): ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes'] ... (total 49)



Encoded feature matrix shapes:


X_train_enc:   (180371, 52)
X_test_enc:    (38651, 52)
X_holdout_enc: (38651, 52)


## 4. Train AdaBoost Classifier

We train an **AdaBoost (Adaptive Boosting)** classifier using decision stumps (`DecisionTreeClassifier(max_depth=1)`) as base weak learners:
- `n_estimators = 50`: Ensemble size.
- `learning_rate = 1.0`: Contribution shrinkage factor per weak learner iteration.
- `random_state = RANDOM_STATE`: Ensures reproducibility.

In [5]:
print("=== Training AdaBoost Classifier on UNSW-NB15 Training Set (70%) ===")
adaboost_clf = AdaBoostClassifier(
    n_estimators=50,
    learning_rate=1.0,
    random_state=RANDOM_STATE
)

t_start = time.time()
adaboost_clf.fit(X_train_enc, y_train_bin)
t_train = time.time() - t_start

print(f"Training completed in {t_train:.2f} seconds ({t_train/60:.2f} minutes).")
print(f"Base estimator: {adaboost_clf.estimator_}")
print(f"Number of estimators: {len(adaboost_clf.estimators_)}")

=== Training AdaBoost Classifier on UNSW-NB15 Training Set (70%) ===


Training completed in 32.78 seconds (0.55 minutes).
Base estimator: DecisionTreeClassifier(max_depth=1)
Number of estimators: 50


## 5. Export Trained Model Artifacts

Export the trained AdaBoost model, feature encoder, and training metadata to the project model registry under:
`models/IDS/adaboost/{timestamp}/`

Directory structure:
- `models/IDS/adaboost/{timestamp}/adaboost_model.pkl`: Serialized AdaBoost classifier.
- `models/IDS/adaboost/{timestamp}/ordinal_encoder.pkl`: Fitted OrdinalEncoder for categorical features.
- `models/IDS/adaboost/{timestamp}/metadata.json`: Model hyperparameters, feature list, and training information.

In [6]:
# Resolve models root directory
possible_models_dirs = [
    base_dir / '../../models',
    base_dir / 'models',
    Path(r'd:/Swinburne Courses/Project A/models')
]
models_root = next((d.resolve() for d in possible_models_dirs if d.resolve().exists()), None)
if models_root is None:
    raise FileNotFoundError("Could not locate models directory.")

# Structure: models/{IDS or MIA}/{name of the model}/{date and time trained the model}
timestamp_str = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
model_category = "IDS"
model_name = "adaboost"
save_dir = models_root / model_category / model_name / timestamp_str
save_dir.mkdir(parents=True, exist_ok=True)

# 1. Export model pickle file
model_file = save_dir / "adaboost_model.pkl"
with open(model_file, "wb") as f:
    pickle.dump(adaboost_clf, f)

# 2. Export feature encoder
encoder_file = save_dir / "ordinal_encoder.pkl"
with open(encoder_file, "wb") as f:
    pickle.dump(encoder, f)

# 3. Export metadata
metadata = {
    "model_category": model_category,
    "model_name": model_name,
    "timestamp": timestamp_str,
    "training_samples": len(X_train_enc),
    "feature_count": len(feature_cols),
    "feature_names": feature_cols,
    "categorical_columns": cat_cols,
    "dataset_source": "datasets/UNSW-NB15_splited",
    "hyperparameters": {
        "n_estimators": 50,
        "learning_rate": 1.0,
        "random_state": RANDOM_STATE,
        "base_estimator": "DecisionTreeClassifier(max_depth=1)"
    },
    "training_time_seconds": round(t_train, 2)
}

metadata_file = save_dir / "metadata.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print(f"=== Model Successfully Exported ===")
print(f"Destination: {save_dir.resolve()}")
print(f"Files saved:")
print(f"  - Model:    {model_file.name} ({model_file.stat().st_size:,} bytes)")
print(f"  - Encoder:  {encoder_file.name} ({encoder_file.stat().st_size:,} bytes)")
print(f"  - Metadata: {metadata_file.name} ({metadata_file.stat().st_size:,} bytes)")

=== Model Successfully Exported ===
Destination: D:\Swinburne Courses\Project A\models\IDS\adaboost\2026-09-28_16-21-01
Files saved:
  - Model:    adaboost_model.pkl (29,565 bytes)
  - Encoder:  ordinal_encoder.pkl (1,798 bytes)
  - Metadata: metadata.json (1,650 bytes)
